In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
root_dir = notebook_dir.parent.parent
sys.path.append(str(root_dir))

root_dir

In [ ]:
from benchmark.configs import Configs

In [ ]:
data_dir = Path.cwd().parent.parent.parent.resolve() / "data"

dataset = "oncloudn"

img_dir = data_dir / f"images/{dataset}"
label_dir = data_dir / f"labels/{dataset}"
csv_dir = data_dir / "metadata"
pred_dir = data_dir / f"predictions/{dataset}"

metadata_path = csv_dir / f"{dataset}_metadata.csv"

In [ ]:
img_dir.mkdir(parents=True, exist_ok=True)
label_dir.mkdir(parents=True, exist_ok=True)
csv_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
from benchmark.core.metadata_io import build_dataset_metadata

df = build_dataset_metadata(
    dataset_name=dataset,
    data_root=data_dir,
    output_csv=metadata_path,
    add_label_distribution=True,
)

In [ ]:
from benchmark.inference import simple
from pathlib import Path
import random

config = Configs.balanced()
config.batch_size = 16
config.use_tta = True
model_weights_path = Path(config.model_name) / Path("assets/cloud_model.pt")
simple(model_weights_path, df.sample(frac=0.01, random_state=42), pred_dir, config)

In [ ]:
from PIL import Image
import xarray
import xrspatial.multispectral as ms

def get_xarray(filepath):
    """Put images in xarray.DataArray format"""
    im_arr = np.array(Image.open(filepath))
    return xarray.DataArray(im_arr, dims=["y", "x"])

def true_color_img(B02_path, B03_path, B04_path):
    """Given the path to the directory of Sentinel-2 chip feature images,
    plots the true color image"""
    red = get_xarray(B04_path)
    green = get_xarray(B03_path)
    blue = get_xarray(B02_path)

    return ms.true_color(r=red, g=green, b=blue, c=5, th=0.3)

In [ ]:
files = list(pred_dir.glob("*.tif"))

pred_files = random.sample(files, len(files))
for pred_file in pred_files:
    B02_file = img_dir / pred_file.stem / "B02.tif"
    B03_file = img_dir / pred_file.stem / "B03.tif"
    B04_file = img_dir / pred_file.stem / "B04.tif"
    label_file = label_dir / pred_file.name

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    if B02_file.exists():
        ax[0].imshow(true_color_img(B02_file, B03_file, B04_file))
        ax[0].set_title(f"True color image for Chip {label_file.stem}")
    if label_file.exists():
        label = Image.open(label_file)
        ax[1].imshow(label)
        ax[1].set_title(f"Chip {label_file.stem} label")
    if pred_file.exists():
        pred = Image.open(pred_file)
        ax[2].imshow(pred)
        pred_im = ax[2].imshow(pred)
        fig.colorbar(pred_im, ax=ax[2])
        ax[2].set_title(f"Predicted Chip {label_file.stem} label")
    plt.tight_layout()
    plt.show()